# LeanCoder Tutorial: Evaluation

This notebook demonstrates how to evaluate a trained model.

In [ ]:
import sys
sys.path.insert(0, '../src')

from eval.benchmarks import BenchmarkSuite
from eval.sandbox import DockerSandbox, MultiTestSandbox
from registry.manager import ModelRegistry
from config import config

## Load Registry and Model

In [ ]:
# Load registry
registry = ModelRegistry(str(config.registry_path))

# Get best checkpoint
best = registry.get_best()

if best:
    print(f"Best checkpoint: {best.id}")
    print(f"  Path: {best.path}")
    if best.eval:
        print(f"  Pass@1: {best.eval.pass_at_1:.2%}")
else:
    print("No checkpoints found")

## Initialize Sandbox

In [ ]:
# Create Docker sandbox
sandbox = DockerSandbox(
    image="python:3.11-slim",
    timeout=10,
    memory_mb=512,
)

# Check if Docker is available
if sandbox.check_docker_available():
    print("Docker sandbox available")
else:
    print("Docker not available, will use fallback")

## Test Code Execution

In [ ]:
# Test code execution
code = """
def add(a, b):
    return a + b
"""

tests = [
    "assert add(1, 2) == 3",
    "assert add(-1, 1) == 0",
    "assert add(0, 0) == 0",
]

result = sandbox.execute(code, tests)

print(f"Success: {result.success}")
print(f"Execution time: {result.execution_time:.2f}s")
if result.error:
    print(f"Error: {result.error}")

## Run Comprehensive Evaluation

In [ ]:
# Create benchmark suite
suite = BenchmarkSuite()

# Load model (placeholder)
model = None

# Run all benchmarks
results = suite.run_all_benchmarks(model, n_samples=10)

# Get summary
summary = suite.get_summary(results)

print("\n" + "="*60)
print("Evaluation Summary")
print("="*60)
print(f"Total Suites: {summary['total_suites']}")
print(f"Total Problems: {summary['total_problems']}")
print(f"Total Passed: {summary['total_passed']}")
print(f"Overall Pass Rate: {summary['overall_pass_rate']:.2%}")

for suite_name, suite_summary in summary['suites'].items():
    print(f"\n{suite_name.upper()}:")
    print(f"  Pass@1: {suite_summary['pass_at_1']:.2%}")
    print(f"  Passed: {suite_summary['passed']}/{suite_summary['total']}")

## Summary

This notebook demonstrated:
1. Loading models from registry
2. Using Docker sandbox for safe code execution
3. Running comprehensive evaluation
4. Analyzing results across multiple benchmarks

Your model is now ready for deployment!